# Ⅱ-02 · 데이터 전처리 실습 — **정답용**

교과서 **72~79쪽**. 빈칸이 모두 채워져 있는 완성본입니다.

> 진도가 빠른 학생, 결석해서 따라가야 하는 학생, 그리고 화면에 띄워 함께 볼 때 씁니다.
> 처음 해 보는 것이라면 **학생용**을 먼저 열어 직접 채워 보세요.

---

1. 맨 위 **파일 → 드라이브에 사본 저장**.
2. 셀 왼쪽 **▶** 또는 **Shift + Enter**.
3. **위에서부터 차례로** 실행하세요.


## 준비 — 판다스 부르고 파일 두 개 읽기

`pandas` 는 **표를 다루는 연장**입니다. 엑셀이라고 생각하면 편합니다.
`as pd` 는 «앞으로 `pd` 라는 짧은 이름으로 부르겠다»는 뜻입니다.

파일은 **선생님 저장소에서 바로** 읽어 옵니다. 내려받거나 올릴 것이 없습니다.

In [ ]:
import pandas as pd

주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'

df1 = pd.read_csv(주소 + 'file1.csv')
df2 = pd.read_csv(주소 + 'file2.csv')

print('df1 :', df1.shape)   # (행 개수, 열 개수)
print('df2 :', df2.shape)

아무것도 안 나온 것 같으면 **`print` 를 안 썼기 때문**입니다.
셀의 **맨 마지막 줄**에 이름만 적으면 그 내용이 표로 예쁘게 나옵니다.

In [ ]:
df1

In [ ]:
df2

## 문제 찾기 — 이 표, 그냥 쓸 수 있을까요?

위 두 표를 눈으로 훑어보세요. **다섯 가지**가 보입니다.

| | 무엇이 | 어디에 |
|---|---|---|
| ① | 같은 사람이 **두 번** | 계약번호 **6번**이 df1 과 df2 에 모두 |
| ② | **288세** | df1 의 3번 |
| ③ | **빈칸**(`NaN`) | df2 의 11번 BMI |
| ④ | 성별 표현이 **제각각** | df1 은 «남/여», df2 는 «남자/여자» |
| ⑤ | 의료비와 **상관없는 열** | 연락처, 그리고 계약번호 |

이것을 치우는 일을 **전처리**라고 하고, 네 가지로 나눕니다 —
**변환 · 통합 · 정제 · 축소**. 아래에서 이 순서대로 합니다.

> **왜 순서가 중요할까요?**
> 표현을 안 맞추고 합치면 «남»과 «남자»를 **따로 세게** 됩니다.
> 중복을 안 지우고 평균을 내면 그 사람이 **두 번 세어집니다.**

---
## ① 변환 — 표현을 같게 맞추기

`replace()` 는 그 열 전체에서 값을 **한꺼번에 바꿔** 줍니다.
`{'바꿀 값':'바뀐 값'}` 모양으로 넘겨 줍니다(**사전**이라고 부릅니다).

> **힌트** · 값을 바꾸는 명령은 «바꾸다»라는 뜻의 영어 낱말입니다.

In [ ]:
df2['성별'] = df2['성별'].replace({'남자':'남', '여자':'여'})

df2

`df2['성별'] = ...` 처럼 **왼쪽에 다시 넣어 준** 것을 잘 보세요.
판다스는 **바꾼 결과를 새로 돌려줄 뿐**, 원래 표를 건드리지 않습니다.
다시 넣어 주지 않으면 **아무 일도 일어나지 않습니다.** 오늘 가장 많이 하는 실수입니다.

---
## ② 통합 — 두 표를 하나로

`pd.concat([표1, 표2])` 는 두 표를 **위아래로** 이어 붙입니다.

> **힌트** · «이어 붙이다(concatenate)»를 줄인 이름입니다.

In [ ]:
df = pd.concat([df1, df2])

print(df.shape)
df

**13행**이 되었지요? 그런데 왼쪽 번호가 0,1,2… 0,1,2… 로 **두 번 돕니다.**
df1 의 번호와 df2 의 번호가 그대로 따라온 것입니다 — 지금은 신경 쓰지 않아도 됩니다.

**계약번호 6번이 두 줄**인 것을 확인하세요. 다음에 치울 것입니다.

---
## ③ 정제 (1) — 중복 지우기

먼저 **정말 중복이 있는지** 확인합니다.
`duplicated()` 가 줄마다 True/False 를 내놓고, `any()` 가 «하나라도 True 가 있나»를 봅니다.

In [ ]:
df['계약번호'].duplicated().any()

`True` 가 나왔으면 중복이 있다는 뜻입니다. 지웁시다.

`keep='first'` 는 «**먼저 나온 줄을 남기고** 뒤엣것을 지우라»는 뜻입니다.

> **힌트** · «중복(duplicates)을 버린다(drop)»를 밑줄로 이은 이름입니다.

In [ ]:
df = df.drop_duplicates(subset='계약번호', keep='first')

print(df.shape)
df

**12행**이 되었으면 맞습니다.

---
## ④ 정제 (2) — 빈칸(결측치) 지우기

빈칸은 **코드로 쉽게 찾습니다.**
`isnull()` 이 칸마다 «비었나?»를 True/False 로 내놓고,
`sum()` 이 그것을 더합니다 — **True 는 1로 세어집니다.**

> **힌트** · 더하는 명령입니다.

In [ ]:
df.isnull().sum()

**BMI 에 1개**가 나왔습니다. 이제 지웁니다.

> **힌트** · «비어 있는 것(na)을 버린다(drop)»입니다.

In [ ]:
print('지우기 전 :', df.shape)

df = df.dropna()

print('지운 뒤   :', df.shape)

**11행**이 되었으면 맞습니다.

> **지우는 것만이 답은 아닙니다.** 다시 조사할 수 있으면 다시 받고,
> 그럴 수 없으면 **평균**이나 **가장 많이 나온 값**으로 채우기도 합니다.
> 빈칸이 **그 자체로 의미 있는** 경우도 있습니다 —
> «지하철 이용자 수»가 어느 날만 0이라면 그건 오류가 아니라 **폭설이나 파업**일 수 있습니다.

---
## ⑤ 정제 (3) — 이상치 지우기

빈칸과 달리 **이상치는 코드가 알아서 못 찾습니다.**
«288세»가 이상한 줄은 **사람이 알아야** 하니까요.

대신 **상자그림(box plot)** 을 그리면 눈에 띕니다.
상자 **바깥에 찍힌 점**이 이상치입니다.

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(df['나이'])
plt.show()

위쪽에 점 하나가 멀찍이 떨어져 있지요? 그게 288세입니다.

«200세를 최대 수명으로 보자»고 정하고 지웁니다. **기준은 사람이 정합니다.**

`df[조건]` 은 «조건이 맞는 줄만 남겨라»라는 뜻입니다.

> **힌트** · «200보다 작거나 같다»를 기호 두 개로 씁니다.

In [ ]:
df = df[df['나이'] <= 200]

print(df.shape)

plt.boxplot(df['나이'])
plt.show()

**10행**이 되고, 상자그림이 정상으로 돌아왔으면 맞습니다.

---
## ⑥ 축소 — 필요 없는 열 덜어 내기

연락처가 의료비와 무슨 상관이 있을까요? 없습니다.
계약번호도 중복을 찾는 데 썼으니 이제 필요 없습니다.

`axis=1` 은 «**열** 단위로 지우라»는 뜻입니다. `axis=0` 이면 행입니다.

> **힌트** · `axis` 에 넣을 숫자 하나입니다.

In [ ]:
df = df.drop('연락처', axis=1)
df = df.drop('계약번호', axis=1)

print(df.shape)
df

**10행 6열.** 13행 9열이던 것이 이만큼 줄었습니다.
이제야 학습에 쓸 수 있는 데이터가 되었습니다.

치운 결과를 파일로 저장해 둡니다. `index=False` 는 «맨 왼쪽 번호는 빼고 저장하라»는 뜻입니다.

In [ ]:
df.to_csv('combined_file.csv', index=False)

print('저장했습니다. 왼쪽 📁 단추를 눌러 보세요.')

---
## 마지막 — 어떤 속성이 의료비와 가장 관련이 깊을까?

이제 **핵심 속성**을 찾습니다.
두 값이 같이 오르내리는 정도를 숫자로 잰 것이 **상관계수**입니다.

- **1** 에 가까우면 → 하나가 오를 때 다른 하나도 오른다
- **-1** 에 가까우면 → 하나가 오를 때 다른 하나는 내린다
- **0** 에 가까우면 → 별 관계가 없다

`numeric_only=True` 는 «숫자로 된 열만 보라»는 뜻입니다(성별·흡연여부는 글자라서 못 셉니다).

In [ ]:
corr = df.corr(numeric_only=True)
corr

의료비 열만 뽑아 크기순으로 봅시다.

In [ ]:
corr['의료비'].sort_values(ascending=False)

**BMI 가 0.75** 로 가장 높습니다. 자녀수 0.20, 나이 0.10 보다 훨씬 크지요.
그래서 «의료비를 예측한다면 **BMI 를 꼭 넣어야겠다**»고 판단합니다.

> **⚠ 함부로 일반화하지 마세요.**
> 이 실습의 표본은 **열 명**입니다. «BMI가 나이보다 의료비와 관련이 깊다»를
> 세상의 사실로 받아들이면 안 됩니다. 교과서도 79쪽에서 같은 말을 합니다.

---
## 다 했습니다 · 확인해 보세요

- [ ] 전처리 네 가지를 말할 수 있다 — **변환 · 통합 · 정제 · 축소**
- [ ] **이상치**(정상 범위를 벗어난 값)와 **결측치**(빈칸)를 구분한다
- [ ] 결측치는 코드로 찾지만, **이상치는 사람이 봐야 한다**
- [ ] `df = df.dropna()` 처럼 **다시 넣어 줘야** 실제로 바뀐다
- [ ] `axis=1` 은 **열**, `axis=0` 은 행
- [ ] 상관계수는 **절댓값이 1에 가까울수록** 관계가 뚜렷하다

### 한 걸음 더 (시간이 남으면)

`df['나이'] <= 200` 의 **200을 50으로** 바꿔 다시 돌려 보세요.
몇 줄이 남나요? 그리고 상관계수는 어떻게 달라지나요?
**기준을 어떻게 잡느냐에 따라 결론이 달라진다**는 것을 눈으로 보게 됩니다.

---
<details>
<summary><b>교과서 방식으로 파일 올리기 (눌러서 펴기)</b></summary>

교과서 73쪽은 파일을 **직접 올립니다.** 시험에 이 방식이 나올 수 있으니 한 번 읽어 두세요.

```python
from google.colab import files
files.upload()          # ← 「파일 선택」 단추가 나옵니다. file1.csv, file2.csv 를 한꺼번에 고릅니다.

df1 = pd.read_csv('file1.csv')
df2 = pd.read_csv('file2.csv')
```

크기가 큰 파일은 올리는 데 오래 걸려서, **구글 드라이브에 올려 두고 연결**하기도 합니다(교과서 80쪽).

```python
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/My Drive/02/02/파일이름.csv', encoding='cp949')
```

`encoding='cp949'` 는 «윈도우에서 만든 한글 파일»을 읽을 때 붙입니다.
이것 없이 읽으면 글자가 깨지거나 오류가 납니다.

**오늘 우리가 쓴 방식**(주소로 바로 읽기)은 올리는 시간이 0초라서 수업에서 씁니다.
셋 다 «표를 읽어 온다»는 점은 똑같습니다.

</details>

막히면 👉 [에러 응급처치 사전](https://richee-pc.github.io/AI_cs/colab.html)

*made by ptp 🐰*